# JOINTS 2026 Blend PoC Notebook

Objective: Membandingkan Kumo, Kumo + LGBM, dan Kumo + LGBM + NN pada tiga lipatan validasi waktu (Juli, Agustus, September 2025), dengan data latih selalu lebih awal dari data validasi dan MASE sebagai metrik evaluasi.

# Sinkronisasi versi modules dan libraries

In [1]:
import sys
print(sys.version)
%pip install -q numpy==2.4.4 pandas==2.3.3 scipy==1.17.1 scikit-learn==1.8.0 lightgbm==4.6.0 torch==2.11.0 tqdm==4.67.3 huggingface_hub==1.24.0

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\leouw\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Import modules dan libraries

In [2]:
import os, glob, random, re
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from lightgbm import LGBMRegressor
from huggingface_hub import snapshot_download
import sdm
from sdm import TableTensor

# Import datasets

In [3]:
_DATA_DIR_CANDIDATES = [
    "../data", "data", "../../data",
    *glob.glob("/kaggle/input/*"), *glob.glob("/kaggle/input/**/", recursive=True),
]
DATA_DIR = next((d for d in _DATA_DIR_CANDIDATES if d and os.path.exists(os.path.join(d, "test_history.csv"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("test_history.csv not found in any candidate DATA_DIR - set it manually.")
print("DATA_DIR =", os.path.abspath(DATA_DIR))

train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["date_show"])
test_hist = pd.read_csv(f"{DATA_DIR}/test_history.csv", parse_dates=["date_show"])
test = pd.read_csv(f"{DATA_DIR}/test.csv", parse_dates=["date_show"])
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
holidays = pd.read_csv(f"{DATA_DIR}/holidays.csv", parse_dates=["date"])
prices = pd.read_csv(f"{DATA_DIR}/ticket_prices.csv")

DATA_DIR = c:\MyFolder\Coding\Data Science\Lomba\JOINTS\JOINTS\data


### External datasets
Kalender publik yang tidak ada pada file holidays bawaan: hari cuti bersama, tanggal Ramadan, libur sekolah, dan libur nasional tambahan, ditambah tabel profil film dan chart box office nasional. Semuanya dipublikasikan pada atau sebelum 2025-09-30, sehingga tidak ada data dari periode test yang dipakai.

In [4]:
_EXTERNAL_DIR_CANDIDATES = ["../external", "external", "../../external", *glob.glob("/kaggle/input/*")]
EXTERNAL_DIR = next((d for d in _EXTERNAL_DIR_CANDIDATES if os.path.exists(os.path.join(d, "cuti_bersama.csv"))), None)
if EXTERNAL_DIR is None:
    raise FileNotFoundError("cuti_bersama.csv not found in any candidate EXTERNAL_DIR - set it manually.")
print("EXTERNAL_DIR =", os.path.abspath(EXTERNAL_DIR))

cuti = pd.read_csv(f"{EXTERNAL_DIR}/cuti_bersama.csv", parse_dates=["date"])
ramadan = pd.read_csv(f"{EXTERNAL_DIR}/ramadan.csv", parse_dates=["date"])
school = pd.read_csv(f"{EXTERNAL_DIR}/school_holidays.csv", parse_dates=["date"])
extra_holidays = pd.read_csv(f"{EXTERNAL_DIR}/extra_national_holidays.csv", parse_dates=["date"])

EXTERNAL_DIR = c:\MyFolder\Coding\Data Science\Lomba\JOINTS\JOINTS\external


# Aggregate Dataset
Mengubah catatan harian menjadi satu baris per pasangan film dan bioskop untuk setiap hari target, dengan format yang sama untuk train dan test.

### Penentuan D1
D1 adalah hari pertama film tayang di setidaknya setengah dari jumlah bioskop pada rilis terluasnya, sehingga preview dan tayang terbatas tidak menentukan D1. Film yang sudah tayang sebelum periode train dimulai, atau yang D10-nya jatuh setelah periode train berakhir, dikeluarkan karena jendela D1-D10 nya tidak lengkap.

In [5]:
D1_MIN_SHARE = 0.5

cinemas_per_day = train.groupby(["movie_title", "date_show"])["cinema_ids"].nunique()
first_show = train.groupby("movie_title")["date_show"].min()
wide_release = cinemas_per_day[cinemas_per_day >= D1_MIN_SHARE * cinemas_per_day.groupby(level=0).transform("max")]
train_d1 = wide_release.reset_index().groupby("movie_title")["date_show"].min()
train_d1 = train_d1[
    (first_show[train_d1.index] > train["date_show"].min())
    & (train_d1 + pd.Timedelta(days=9) <= train["date_show"].max())
]
test_d1 = test_hist.groupby("movie_title")["date_show"].min()

print(f"film train: {len(train_d1)} | film test_history: {len(test_d1)} | film test: {test['movie_title'].nunique()}")

film train: 205 | film test_history: 163 | film test: 160


### Pasangan film dan bioskop
Membangun jendela D1-D10 untuk setiap pasangan dengan tiket dan jumlah show per hari. D1-D3 menjadi input dan D4-D10 menjadi target; pasangan test hanya memiliki D1-D3. Pasangan train tanpa penjualan pada D3 dibuang, sama seperti definisi pasangan test.

In [6]:
DAILY_COLS = {"total_ticket": "t", "total_show": "show"}

def build_window(hist, d1_map, n_days):
    df = hist[hist["movie_title"].isin(d1_map.index)].copy()
    df["day"] = (df["date_show"] - df["movie_title"].map(d1_map)).dt.days + 1
    df = df[df["day"].between(1, n_days)]
    parts = []
    for col, prefix in DAILY_COLS.items():
        p = df.pivot_table(index=["movie_title", "cinema_ids"], columns="day", values=col, fill_value=0)
        p = p.reindex(columns=range(1, n_days + 1), fill_value=0)
        p.columns = [f"{prefix}{d}" for d in p.columns]
        parts.append(p)
    out = pd.concat(parts, axis=1).reset_index()
    out["d1_date"] = out["movie_title"].map(d1_map)
    return out

train_window = build_window(train, train_d1, 10)
test_window = build_window(test_hist, test_d1, 3)

train_pairs = train_window[train_window["t3"] > 0].reset_index(drop=True)
test_pairs = test_window.merge(test[["movie_title", "cinema_ids"]].drop_duplicates(), on=["movie_title", "cinema_ids"])
print(f"pasangan train: {len(train_pairs)} | pasangan test: {len(test_pairs)}")

pasangan train: 8007 | pasangan test: 10373


### Format harian
Satu baris per pasangan dan hari target (horizon 4 sampai 10), sehingga satu model memprediksi semua horizon dan horizon menjadi fitur.

In [7]:
HORIZONS = range(4, 11)
INPUT_COLS = [f"{p}{d}" for p in DAILY_COLS.values() for d in (1, 2, 3)]

def to_long(pairs, with_target):
    rows = []
    for h in HORIZONS:
        r = pairs[["movie_title", "cinema_ids", "d1_date"] + INPUT_COLS].copy()
        r["horizon"] = h
        r["date_show"] = r["d1_date"] + pd.Timedelta(days=h - 1)
        r["total_ticket"] = pairs[f"t{h}"].values if with_target else np.nan
        rows.append(r)
    return pd.concat(rows, ignore_index=True)

train_long = to_long(train_pairs, with_target=True).assign(is_train=1)
test_long = to_long(test_pairs, with_target=False).assign(is_train=0).merge(
    test[["id", "movie_title", "cinema_ids", "date_show"]], on=["movie_title", "cinema_ids", "date_show"], how="left"
)
full = pd.concat([train_long, test_long], ignore_index=True)
print(f"baris train: {len(train_long)} | baris test: {len(test_long)} | baris test.csv: {len(test)}")

baris train: 56049 | baris test: 72611 | baris test.csv: 72611


### Penggabungan sumber data
Menambahkan kota tiap bioskop, tipe hari libur, batas harga tiket, dan metadata film. Akhiran 2D, 3D, dan IMAX dihapus dari judul sebelum penggabungan metadata, karena semua versi format dari satu film memiliki informasi film yang sama.

In [8]:
FORMAT_PATTERN = r"\s*\((?:IMAX )?[23]D\)\s*$"
CINEMA_CITY = pd.concat([train, test_hist]).drop_duplicates("cinema_ids").set_index("cinema_ids")["city_name"]
PRICE_DAY = {"weekday": "Weekday", "friday": "Friday", "weekend": "Weekend"}

def join_sources(df):
    n = len(df)
    df = df.copy()
    df["city_name"] = df["cinema_ids"].map(CINEMA_CITY)
    df = df.merge(holidays.rename(columns={"date": "date_show"}), on="date_show", how="left")
    df["price_day"] = df["day_tipe"].map(PRICE_DAY)
    df = df.merge(prices.rename(columns={"ceil": "price"}), on=["city_name", "price_day"], how="left")
    df["base_title"] = df["movie_title"].str.replace(FORMAT_PATTERN, "", regex=True)
    df = df.merge(movies.rename(columns={"original_title": "base_title"}), on="base_title", how="left")
    assert len(df) == n
    return df

full = join_sources(full)
print(f"full: {full.shape}")
full.isna().sum()[lambda s: s > 0]

full: (128660, 27)


total_ticket     72611
id               56049
holiday_name    122732
dtype: int64

### Penggabungan external datasets
Menambahkan cuti bersama dan libur sekolah ke setiap baris berdasarkan tanggal.

In [9]:
def join_external(df):
    n = len(df)
    df = df.merge(cuti[["date", "name"]].rename(columns={"date": "date_show", "name": "cuti_bersama_name"}), on="date_show", how="left")
    df = df.merge(school[["date", "period"]].rename(columns={"date": "date_show", "period": "school_break"}), on="date_show", how="left")
    assert len(df) == n
    return df

full = join_external(full)
print(f"full: {full.shape}")
full.groupby("is_train")[["cuti_bersama_name", "school_break"]].count()

full: (128660, 29)


,cuti_bersama_name,school_break
is_train,,
0,2392,12540
1,1141,6208


In [10]:
full.head()

,movie_title,cinema_ids,d1_date,t1,t2,t3,show1,show2,show3,horizon,...,price,base_title,age_rating,genre,producer,director,writer,casts,cuti_bersama_name,school_break
0,28 YEARS LATER,015555fbcd8faef6b55a854605df0a87,2025-06-18,261.0,210.0,137.0,18.0,17.0,12.0,4,...,45000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
1,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,2025-06-18,1376.0,1033.0,946.0,23.0,23.0,23.0,4,...,45000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
2,28 YEARS LATER,04a2de432a8901363a05137a8e8de1d2,2025-06-18,0.0,83.0,85.0,0.0,3.0,3.0,4,...,40000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
3,28 YEARS LATER,073b764928864ed8fa0b6b115bfdd685,2025-06-18,159.0,55.0,43.0,9.0,3.0,2.0,4,...,45000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
4,28 YEARS LATER,108f9a4d181f12fd1b7e3c9adae38478,2025-06-18,643.0,458.0,447.0,30.0,19.0,20.0,4,...,55000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN


# Preprocessing

### Hapus hari yang tidak lengkap
Hari ketika train.csv memiliki baris jauh lebih sedikit dari hari-hari di sekitarnya adalah rekaman parsial. Baris train yang hari targetnya atau jendela D1-D3 nya menyentuh hari seperti itu dihapus, karena tiketnya tercatat terlalu rendah.

In [11]:
rows_per_day = train.groupby("date_show").size().reindex(
    pd.date_range(train["date_show"].min(), train["date_show"].max()), fill_value=0
)
row_share = rows_per_day / rows_per_day.rolling(15, center=True, min_periods=5).median()
GAP_DAYS = row_share.index[row_share < 0.7]

def drop_gap_rows(df):
    d13_on_gap = np.any([(df["d1_date"] + pd.Timedelta(days=k)).isin(GAP_DAYS) for k in range(3)], axis=0)
    broken = (df["is_train"] == 1) & (df["date_show"].isin(GAP_DAYS) | d13_on_gap)
    return df[~broken].reset_index(drop=True)

print("gap days:", [d.date().isoformat() for d in GAP_DAYS])
n_before = len(full)
full = drop_gap_rows(full)
print(f"dropped train rows: {n_before - len(full)} | full: {full.shape}")

gap days: ['2025-06-07', '2025-06-09', '2025-06-11', '2025-06-13', '2025-06-14', '2025-06-15', '2025-06-16']
dropped train rows: 1378 | full: (127282, 29)


### Flag kalender
Hari libur membuat lebih banyak orang datang ke bioskop. File holidays hanya menandai libur nasional, sedangkan cuti bersama dan libur sekolah berasal dari kalender eksternal, sehingga masing-masing diberi flag sendiri.

In [12]:
def add_calendar_flags(df):
    df = df.copy()
    df["is_holiday"] = (df["holiday_tipe"] == "holiday").astype(int)
    df["is_cuti_bersama"] = df["cuti_bersama_name"].notna().astype(int)
    df["is_school_holiday"] = df["school_break"].notna().astype(int)
    return df

full = add_calendar_flags(full)
full.groupby("is_train")[["is_holiday", "is_cuti_bersama", "is_school_holiday"]].mean().round(3)

,is_holiday,is_cuti_bersama,is_school_holiday
is_train,,,
0,0.041,0.033,0.173
1,0.054,0.018,0.114


### Kolom genre
Genre mengubah seberapa cepat penjualan film turun: film keluarga dan animasi bertahan lebih lama, sedangkan film horor cepat turun. Setiap genre menjadi satu kolom 0 atau 1, dan hanya genre yang muncul pada setidaknya 2% baris train yang dipakai, karena genre yang lebih jarang terlalu sedikit barisnya untuk dipelajari model.

In [13]:
def add_genres(df, genre_cols=None):
    dummies = df["genre"].str.get_dummies(sep=", ").add_prefix("genre_")
    dummies.columns = dummies.columns.str.replace(" ", "_")
    if genre_cols is not None:
        dummies = dummies.reindex(columns=genre_cols, fill_value=0)
    return pd.concat([df, dummies], axis=1), list(dummies.columns)

full, GENRE_COLS = add_genres(full)
COMMON_GENRES = [g for g in GENRE_COLS if full.loc[full["is_train"] == 1, g].mean() >= 0.02]
print(f"genres: {len(GENRE_COLS)} | used as features: {len(COMMON_GENRES)}")
full.loc[full["is_train"] == 1, GENRE_COLS].mean().sort_values(ascending=False).round(3).head(10)

genres: 27 | used as features: 15


genre_Horror       0.375
genre_Drama        0.328
genre_Action       0.212
genre_Thriller     0.146
genre_Comedy       0.120
genre_Animation    0.070
genre_Mystery      0.063
genre_Fantasy      0.056
genre_Superhero    0.052
genre_Family       0.043
dtype: float64

### Kolom kategori
city_name, day_tipe, dan age_rating dideklarasikan sebagai kategori dengan level yang sama pada train dan test, sehingga tree memperlakukannya sebagai kelompok dan bukan angka berurutan, dan kodenya sama antara train dan test.

In [14]:
CAT_COLS = ["city_name", "day_tipe", "age_rating"]
CAT_LEVELS = {col: full[col].astype("category").cat.categories for col in CAT_COLS}

def set_categories(df):
    df = df.copy()
    for col in CAT_COLS:
        df[col] = pd.Categorical(df[col], categories=CAT_LEVELS[col])
    return df

full = set_categories(full)
full[CAT_COLS].nunique()

city_name     69
day_tipe       3
age_rating     4
dtype: int64

### Hapus kolom tanpa sinyal
Kolom kredit (producer, director, writer, casts) hampir unik untuk setiap film sehingga model hanya bisa menghafalnya. Kolom lain yang dihapus hanya dipakai untuk penggabungan atau mengulang informasi yang sudah ter-encode.

In [15]:
DROP_COLS = [
    "holiday_tipe", "holiday_name", "cuti_bersama_name", "school_break", "genre",
    "price_day", "base_title", "producer", "director", "writer", "casts",
]

def drop_unused(df):
    return df.drop(columns=DROP_COLS)

full = drop_unused(full)
print(f"full: {full.shape}")
print(list(full.columns))

full: (127282, 48)
['movie_title', 'cinema_ids', 'd1_date', 't1', 't2', 't3', 'show1', 'show2', 'show3', 'horizon', 'date_show', 'total_ticket', 'is_train', 'id', 'city_name', 'day_tipe', 'price', 'age_rating', 'is_holiday', 'is_cuti_bersama', 'is_school_holiday', 'genre_Action', 'genre_Adventure', 'genre_Animation', 'genre_Biography', 'genre_Comedy', 'genre_Crime', 'genre_Disaster', 'genre_Documentary', 'genre_Drama', 'genre_Family', 'genre_Fantasy', 'genre_Gothic', 'genre_History', 'genre_Horror', 'genre_Music', 'genre_Musical', 'genre_Mystery', 'genre_Religi', 'genre_Romance', 'genre_Sci-Fi', 'genre_Sport', 'genre_Superhero', 'genre_Survival', 'genre_Suspense', 'genre_Teen', 'genre_Thriller', 'genre_War']


### Target
target_ratio adalah tiket pada hari target dibagi scale, yaitu rata-rata D1-D3 (minimal 1). Metrik kompetisi adalah selisih absolut dibagi scale yang sama, sehingga mean absolute error dari target_ratio adalah metriknya sendiri, dan pasangan dengan ukuran yang sangat berbeda menjadi sebanding.

In [16]:
def add_target(df):
    df = df.copy()
    df["scale"] = df[["t1", "t2", "t3"]].mean(axis=1).clip(lower=1)
    df["target_ratio"] = df["total_ticket"] / df["scale"]
    return df

full = add_target(full)
full.loc[full["is_train"] == 1].groupby("horizon")["target_ratio"].median().round(3)

horizon
4     0.915
5     0.616
6     0.467
7     0.360
8     0.163
9     0.000
10    0.000
Name: target_ratio, dtype: float64

# Feature Engineering

### Blok hari libur
Penonton tetap banyak selama rangkaian hari libur berturut-turut dan turun ketika rangkaian itu berakhir, sehingga long weekend berperilaku berbeda dari akhir pekan biasa.
- off_block_len: panjang rangkaian hari libur tanpa putus (akhir pekan, libur nasional, cuti bersama) yang memuat hari target, bernilai 0 untuk hari kerja.

In [17]:
cal = pd.DataFrame({"date": pd.date_range(full["d1_date"].min() - pd.Timedelta(days=30), full["date_show"].max())})
NATIONAL_HOLIDAYS = pd.concat([holidays.loc[holidays["holiday_tipe"] == "holiday", "date"], extra_holidays["date"]])
cal["off"] = (
    (cal["date"].dt.dayofweek >= 5) | cal["date"].isin(NATIONAL_HOLIDAYS) | cal["date"].isin(cuti["date"])
).astype(int)
block_id = (cal["off"] != cal["off"].shift()).cumsum()
cal["block_len"] = cal.groupby(block_id)["off"].transform("size") * cal["off"]
BLOCK_LEN = cal.set_index("date")["block_len"]

def add_off_block(df):
    return df.assign(off_block_len=df["date_show"].map(BLOCK_LEN))

full = add_off_block(full)

print("longest blocks:")
print(cal[cal["block_len"] >= 4].groupby(block_id)["date"].agg(["min", "max", "size"]).reset_index(drop=True))
full.groupby("is_train")["off_block_len"].value_counts(normalize=True).unstack().round(3)

longest blocks:
         min        max  size
0 2025-03-28 2025-04-07    11
1 2025-05-10 2025-05-13     4
2 2025-05-29 2025-06-01     4
3 2025-06-06 2025-06-09     4
4 2025-12-25 2025-12-28     4
5 2026-02-14 2026-02-17     4
6 2026-03-18 2026-03-24     7


off_block_len,0,1,2,3,4,7,11
is_train,,,,,,,
0,0.658,0.005,0.228,0.019,0.047,0.042,NaN
1,0.657,0.006,0.213,0.072,0.051,NaN,0.001


### Fitur pasangan
Perilaku pasangan itu sendiri pada D1-D3, informasi terkuat untuk hari-hari berikutnya:
- trend: tiket D3 relatif terhadap D1, apakah permintaan pasangan naik atau turun.
- t3_share: porsi tiket D1-D3 yang terjual pada D3, menandakan puncak yang terlambat atau mulai tayang terlambat.
- show_mean: rata-rata jumlah show per hari, kapasitas yang diberikan bioskop untuk film tersebut.
- tickets_per_show3: tiket per show pada D3, permintaan yang tidak bergantung pada jumlah show yang dijadwalkan.
- active_days: jumlah hari dengan penjualan pada D1-D3, pasangan yang mulai terlambat memiliki lebih sedikit.

In [18]:
D13 = ["t1", "t2", "t3"]

def add_pair_features(df):
    df = df.copy()
    df["trend"] = (df["t3"] + 1) / (df["t1"] + 1)
    df["t3_share"] = df["t3"] / df[D13].sum(axis=1)
    df["show_mean"] = df[["show1", "show2", "show3"]].mean(axis=1)
    df["tickets_per_show3"] = df["t3"] / df["show3"].replace(0, np.nan)
    df["active_days"] = (df[D13] > 0).sum(axis=1)
    return df

full = add_pair_features(full)
full[["trend", "t3_share", "show_mean", "tickets_per_show3", "active_days"]].describe().round(2)

,trend,t3_share,show_mean,tickets_per_show3,active_days
count,127282.00,127282.00,127282.00,127282.00,127282.00
mean,11.13,0.33,8.34,32.42,2.90
std,51.19,0.16,11.10,31.17,0.36
min,0.01,0.00,0.33,0.25,1.00
25%,0.56,0.24,3.33,12.14,3.00
50%,0.89,0.32,5.00,24.00,3.00
75%,1.49,0.40,9.00,42.20,3.00
max,1199.00,1.00,216.00,421.33,3.00


### Fitur film
Film yang sama di semua bioskop, dijumlahkan dari D1-D3. Film diprediksi lebih baik dari gambaran nasionalnya daripada dari satu bioskop saja.
- mv_t1, mv_t2, mv_t3: tiket nasional film pada masing-masing D1-D3, kekuatan film.
- mv_clusters: jumlah bioskop yang memiliki penjualan pada D3, luasnya rilis.
- mv_trend: D3 nasional relatif terhadap D1, tren yang lebih stabil daripada tren satu pasangan.
- pair_share: porsi tiket D1-D3 nasional film yang dijual oleh bioskop ini, seberapa penting bioskop tersebut bagi film.

In [19]:
def film_features(window):
    g = window.groupby("movie_title")
    out = g[D13].sum().add_prefix("mv_")
    out["mv_clusters"] = g["t3"].apply(lambda s: (s > 0).sum())
    out["mv_trend"] = (out["mv_t3"] + 1) / (out["mv_t1"] + 1)
    return out

FILM_COLS = ["mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend"]

def add_film_features(df, window):
    df = df.copy()
    feats = film_features(window)
    for col in FILM_COLS:
        df[col] = df["movie_title"].map(feats[col]).values
    df["pair_share"] = df[D13].sum(axis=1) / df[["mv_t1", "mv_t2", "mv_t3"]].sum(axis=1)
    return df

is_tr = full["is_train"] == 1
full = pd.concat([add_film_features(full[is_tr], train_window), add_film_features(full[~is_tr], test_window)]).sort_index()
full.groupby("is_train")[FILM_COLS + ["pair_share"]].median().round(3)

,mv_t1,mv_t2,mv_t3,mv_clusters,mv_trend,pair_share
is_train,,,,,,
0,11615.0,12300.0,12249.0,87.0,0.869,0.008
1,22042.0,18511.0,17704.0,81.0,0.923,0.010


### Fitur bioskop
Ukuran dan program tayang yang biasa dari bioskop, dihitung dari jendela D1-D3 semua film tanpa informasi target:
- cl_scale: rata-rata scale D1-D3 film yang tayang, bioskop besar menurun berbeda dari bioskop kecil.
- cl_movies: jumlah film berbeda yang tayang, berapa banyak film yang bersaing untuk layar yang sama.

In [20]:
all_windows = pd.concat([train_window, test_window], ignore_index=True)
cinema_feats = (
    all_windows.assign(pair_scale=all_windows[D13].mean(axis=1))
    .groupby("cinema_ids")
    .agg(cl_scale=("pair_scale", "mean"), cl_movies=("movie_title", "nunique"))
)

def add_cinema_features(df):
    return df.join(cinema_feats, on="cinema_ids")

full = add_cinema_features(full)
cinema_feats.describe().round(2)

,cl_scale,cl_movies
count,121.00,121.00
mean,192.95,192.13
std,165.41,73.64
min,18.77,5.00
25%,106.63,145.00
50%,160.53,189.00
75%,221.98,239.00
max,1348.26,347.00


### Fitur hari dan harga
- dow: hari dalam seminggu dari hari target, akhir pekan lebih ramai.
- d1_dow: hari dalam seminggu dari D1, posisi akhir pekan di dalam D1-D10 mengubah bentuk penurunan.
- price_ratio: harga tiket pada hari target relatif terhadap rata-rata harga D1-D3, hari yang lebih murah menarik lebih banyak orang.

In [21]:
DAY_TIPE_BY_DATE = holidays.set_index("date")["day_tipe"]
PRICE_MAP = prices.set_index(["city_name", "price_day"])["ceil"]

def add_weekday_price(df):
    df = df.copy()
    df["dow"] = df["date_show"].dt.dayofweek
    df["d1_dow"] = df["d1_date"].dt.dayofweek

    def price_on(dates):
        keys = pd.MultiIndex.from_arrays([df["city_name"].astype(str), dates.map(DAY_TIPE_BY_DATE).map(PRICE_DAY)])
        return pd.Series(PRICE_MAP.reindex(keys).values, index=df.index)

    d13_price = pd.concat([price_on(df["d1_date"] + pd.Timedelta(days=k)) for k in range(3)], axis=1).mean(axis=1)
    df["price_ratio"] = df["price"] / d13_price
    return df

full = add_weekday_price(full)
full[["dow", "d1_dow", "price_ratio"]].describe().round(3)

,dow,d1_dow,price_ratio
count,127282.0,127282.000,127282.000
mean,3.0,2.772,0.977
std,2.0,0.726,0.144
min,0.0,0.000,0.615
25%,1.0,2.000,0.875
50%,3.0,3.000,0.955
75%,5.0,3.000,1.105
max,6.0,6.000,1.500


### Hari libur di sekitar jendela D1-D3 dan hari target
Penjualan D1-D3 terdongkrak bila jatuh pada hari libur, dan hari target bisa berada pada situasi kalender yang berbeda, sehingga model perlu tahu keduanya:
- off_d1, off_d2, off_d3: apakah masing-masing hari pada jendela adalah hari libur.
- win_off, win_hol, win_wk: jumlah hari libur, libur nasional, dan hari akhir pekan pada jendela.
- tgt_off: apakah hari target adalah hari libur.
- off_rel: hari target libur dikurangi porsi hari libur pada jendela, jendela yang penuh hari libur lalu diikuti hari kerja memprediksi penurunan.
- tgt_prev_off, tgt_next_off: apakah hari di samping hari target libur, tepi dari long weekend.
- between_off, between_off_share: jumlah dan porsi hari libur antara D4 dan hari target.

In [22]:
def is_off(dates):
    return ((dates.dt.dayofweek >= 5) | dates.isin(NATIONAL_HOLIDAYS) | dates.isin(cuti["date"])).astype(int)

def add_window_offdays(df):
    """days off (weekend, national holiday, cuti bersama) in the D1-D3 window, on and around the target day, and between D4 and the target"""
    df = df.copy()
    day = lambda k: df["d1_date"] + pd.Timedelta(days=k)
    for k in range(3):
        df[f"off_d{k + 1}"] = is_off(day(k))
    df["win_off"] = df[["off_d1", "off_d2", "off_d3"]].sum(axis=1)
    df["win_hol"] = sum(day(k).isin(NATIONAL_HOLIDAYS).astype(int) for k in range(3))
    df["win_wk"] = sum((day(k).dt.dayofweek >= 5).astype(int) for k in range(3))
    df["tgt_off"] = is_off(df["date_show"])
    df["off_rel"] = df["tgt_off"] - df["win_off"] / 3
    df["tgt_prev_off"] = is_off(df["date_show"] - pd.Timedelta(days=1))
    df["tgt_next_off"] = is_off(df["date_show"] + pd.Timedelta(days=1))
    df["between_off"] = sum(((df["horizon"] > h) & (is_off(day(h - 1)) == 1)).astype(int) for h in range(4, 11))
    df["between_off_share"] = df["between_off"] / (df["horizon"] - 4).clip(lower=1)
    return df

OFF_COLS = ["off_d1", "off_d2", "off_d3", "win_off", "win_hol", "win_wk", "tgt_off", "off_rel",
            "tgt_prev_off", "tgt_next_off", "between_off", "between_off_share"]
full = add_window_offdays(full)
full.groupby("is_train")[OFF_COLS].mean().round(3)

,off_d1,off_d2,off_d3,win_off,win_hol,win_wk,tgt_off,off_rel,tgt_prev_off,tgt_next_off,between_off,between_off_share
is_train,,,,,,,,,,,,
0,0.078,0.290,0.692,1.059,0.178,0.723,0.342,-0.011,0.353,0.342,1.200,0.425
1,0.032,0.318,0.749,1.098,0.241,0.838,0.343,-0.023,0.345,0.340,1.118,0.404


### Profil film
Jenis film dari external/film_profile.csv, karena penonton film horor lokal, film asing blockbuster, dan sekuel bertahan berbeda:
- is_local: film Indonesia atau asing.
- is_sequel: bagian dari sekuel atau franchise, yang membawa penonton yang sudah ada.
- kind_code: live action, animasi, anime, rilis ulang, konser, atau dokumenter.
- source_code: cerita orisinal, novel, komik, remake, kisah nyata, atau sumber lain.
- origin_code: negara asal.
- predecessor_adm: jumlah penonton film sebelumnya dalam franchise, ukuran penonton yang sudah ada.
- local_horror: lokal dan horor sekaligus, genre yang dibuka kuat dan cepat turun.
Fakta yang baru dipublikasikan setelah 2025-09-30 dikosongkan, sehingga tidak ada informasi setelah cutoff yang masuk ke model.

In [23]:
PROFILE_CUTOFF = pd.Timestamp("2025-09-30")
FILM_KINDS = ["live_action", "animation", "anime", "rerelease", "concert", "documentary"]
SOURCES = ["original", "novel", "other", "comic", "remake", "true_story", "viral_story", "folklore", "game"]
ORIGINS = ["Indonesia", "United States", "South Korea", "Japan", "Thailand"]

def load_film_profile():
    p = pd.read_csv(f"{EXTERNAL_DIR}/film_profile.csv")
    late = (p["split"] == "test") & ((p["info_public_before_cutoff"] == "no") | (pd.to_datetime(p["fact_source_date"], format="mixed", errors="coerce") > PROFILE_CUTOFF))
    pred_known = pd.to_datetime(p["predecessor_published"], errors="coerce") <= PROFILE_CUTOFF
    first_origin = p["origin_country"].astype(str).str.split(",").str[0].str.strip()
    prof = pd.DataFrame({
        "movie_title": p["movie_title"],
        "is_local": p["is_local"].astype(float),
        "is_sequel": p["is_sequel_or_franchise"].astype(float),
        "kind_code": p["film_kind"].map({k: i for i, k in enumerate(FILM_KINDS)}).astype(float),
        "source_code": p["source_material"].map({k: i for i, k in enumerate(SOURCES)}).astype(float),
        "origin_code": first_origin.map({k: i for i, k in enumerate(ORIGINS)}).fillna(len(ORIGINS)).astype(float),
        "predecessor_adm": np.where(pred_known, np.log1p(p["predecessor_admissions_id"]), np.nan),
    })
    prof.loc[late.values, prof.columns[1:]] = np.nan
    return prof.drop_duplicates("movie_title").set_index("movie_title")

FILM_PROFILE = load_film_profile()
PROFILE_COLS = list(FILM_PROFILE.columns) + ["local_horror"]

def add_film_profile(df):
    df = df.copy()
    for c in FILM_PROFILE.columns:
        df[c] = df["movie_title"].map(FILM_PROFILE[c])
    df["local_horror"] = df["is_local"] * df["genre_Horror"]
    return df

full = add_film_profile(full)
print("films without a profile:", full.loc[full["is_local"].isna()].groupby("is_train")["movie_title"].nunique().to_dict())
full.groupby("is_train")[PROFILE_COLS].mean().round(3)

films without a profile: {0: 8}


,is_local,is_sequel,kind_code,source_code,origin_code,predecessor_adm,local_horror
is_train,,,,,,,
0,0.529,0.337,0.099,1.197,1.229,14.955,0.206
1,0.564,0.309,0.135,2.052,0.753,14.545,0.286


### Pasangan yang mulai terlambat
Sebuah bioskop bisa menambahkan film setelah D1, sehingga pasangan dengan penjualan hanya pada D3 lebih muda daripada yang ditunjukkan horizonnya.
- first_active: hari pertama pada D1-D3 yang memiliki penjualan.
- eff_age: horizon dikurangi first_active ditambah 1, usia sebenarnya dari pasangan pada hari target.
- last_ratio: penjualan pada hari aktif terakhir relatif terhadap scale, level tempat pasangan mengakhiri D1-D3.
- active_mean: rata-rata penjualan hanya pada hari aktif, karena rata-rata D1-D3 meremehkan pasangan yang mulai terlambat.

In [24]:
def add_late_start(df):
    df = df.copy()
    t = df[D13].values
    first = np.select([t[:, 0] > 0, t[:, 1] > 0, t[:, 2] > 0], [1, 2, 3], 4)
    last = np.select([t[:, 2] > 0, t[:, 1] > 0], [t[:, 2], t[:, 1]], t[:, 0])
    df["first_active"] = first
    df["eff_age"] = df["horizon"] - first + 1
    df["last_ratio"] = last / df["scale"]
    df["active_mean"] = t.sum(axis=1) / np.clip((t > 0).sum(axis=1), 1, None) / df["scale"]
    return df

full = add_late_start(full)
full.groupby("is_train")["first_active"].value_counts(normalize=True).unstack().round(3)

first_active,1,2,3
is_train,,,
0,0.916,0.066,0.018
1,0.928,0.055,0.018


### Persaingan dari film lain yang baru rilis
Sebuah film kehilangan layar dan penonton ke film yang rilis setelahnya, dan hari pasar yang kuat mengangkat semua film. Fitur ini hanya memakai informasi D1-D3 film lain, tidak pernah targetnya.
- comp_size_cin: ukuran film yang rilis di bioskop yang sama setelah D1 film ini dan sampai hari target, relatif terhadap scale film itu sendiri.
- comp_fresh_cin: sama, tetapi hanya untuk film dalam tiga hari pertamanya, persaingan layar yang paling kuat.
- comp_size_nat: hitungan yang sama pada tingkat nasional.
- date_idx: seberapa kuat pembukaan film lain pada tanggal target dibandingkan bentuk pembukaan yang biasa untuk hari rilisnya, ukuran seberapa ramai pasar hari itu.
- own_idx: ukuran yang sama pada hari D1-D3 film itu sendiri.
- date_idx_rel: date_idx dikurangi own_idx, apakah hari target lebih ramai atau lebih sepi dari hari-hari saat film diamati.

In [25]:
COMP_MIN_SIZE = 50

def opening_pairs(df):
    return df.drop_duplicates(["movie_title", "cinema_ids"])[["movie_title", "cinema_ids", "d1_date", "t1", "t2", "t3", "scale"]]

def film_openings(df):
    f = opening_pairs(df).groupby(["movie_title", "d1_date"])[D13].sum().reset_index()
    f["m"] = f[D13].mean(axis=1)
    return f[f["m"] >= COMP_MIN_SIZE].reset_index(drop=True)

def typical_opening_shape(df):
    """usual film D_k / mean(D1-D3), k = 1..3, by release weekday"""
    f = film_openings(df)
    log_r = lambda g, k: np.average(np.log((g[f"t{k}"] + 1) / (g["m"] + 1)), weights=np.sqrt(g["m"]))
    overall = [np.exp(log_r(f, k)) for k in (1, 2, 3)]
    by_dow = {dw: [np.exp(log_r(g, k)) for k in (1, 2, 3)] for dw, g in f.groupby(f["d1_date"].dt.dayofweek) if len(g) >= 5}
    return {dw: by_dow.get(dw, overall) for dw in range(7)}

def opening_surprise(df, shape):
    """log of each film's D1-D3 days vs the usual shape for its release weekday, one row per film and date"""
    rows = []
    for r in film_openings(df).itertuples():
        t = shape[r.d1_date.dayofweek]
        for k in range(3):
            rows.append((r.movie_title, r.d1_date + pd.Timedelta(days=k), np.log((getattr(r, f"t{k + 1}") + 1) / (r.m + 1) / t[k]), np.sqrt(r.m)))
    s = pd.DataFrame(rows, columns=["movie_title", "date", "s", "w"])
    return s.assign(sw=s["s"] * s["w"])

def date_strength(dates, titles, surprise):
    """size-weighted mean surprise of the other films in their D1-D3 on each date (NaN when there are none)"""
    tot = surprise.groupby("date")[["sw", "w"]].sum().reindex(pd.DatetimeIndex(dates)).values
    own = surprise.set_index(["date", "movie_title"])[["sw", "w"]].reindex(pd.MultiIndex.from_arrays([pd.DatetimeIndex(dates), titles])).fillna(0).values
    w = tot[:, 1] - own[:, 1]
    ok = w > 1e-9
    return np.where(ok, (tot[:, 0] - own[:, 0]) / np.where(ok, w, 1), np.nan)

def add_competition(df, ref, shape):
    """ref: rows whose pairs are the other films' openings"""
    df = df.reset_index(drop=True)
    others = opening_pairs(ref)
    own_size = opening_pairs(df).groupby("movie_title")["scale"].sum()

    key = df[["movie_title", "cinema_ids", "d1_date", "date_show"]].reset_index()
    m = key.merge(others[["cinema_ids", "movie_title", "d1_date", "scale"]], on="cinema_ids", suffixes=("", "_o"))
    m = m[(m["movie_title_o"] != m["movie_title"]) & (m["d1_date_o"] > m["d1_date"]) & (m["d1_date_o"] <= m["date_show"])]
    fresh = m[(m["date_show"] - m["d1_date_o"]).dt.days <= 2].groupby("index")["scale"].sum()
    df["comp_size_cin"] = np.log1p(m.groupby("index")["scale"].sum().reindex(df.index, fill_value=0).values / df["scale"].values)
    df["comp_fresh_cin"] = np.log1p(fresh.reindex(df.index, fill_value=0).values / df["scale"].values)

    films = others.groupby("movie_title").agg(fd1=("d1_date", "min"), fsize=("scale", "sum")).rename_axis("movie_o").reset_index()
    u = df[["movie_title", "d1_date", "date_show"]].drop_duplicates()
    mm = u.merge(films, how="cross")
    mm = mm[(mm["movie_o"] != mm["movie_title"]) & (mm["fd1"] > mm["d1_date"]) & (mm["fd1"] <= mm["date_show"])]
    u = u.merge(mm.groupby(["movie_title", "d1_date", "date_show"]).agg(comp_size_nat=("fsize", "sum")).reset_index(),
                how="left", on=["movie_title", "d1_date", "date_show"])

    surprise = opening_surprise(ref, shape)
    u["date_idx"] = date_strength(u["date_show"], u["movie_title"].values, surprise)
    own_days = np.vstack([date_strength(u["d1_date"] + pd.Timedelta(days=k), u["movie_title"].values, surprise) for k in range(3)])
    n_days = (~np.isnan(own_days)).sum(axis=0)
    u["own_idx"] = np.where(n_days > 0, np.nansum(own_days, axis=0) / np.maximum(n_days, 1), np.nan)
    u["date_idx_rel"] = u["date_idx"] - u["own_idx"]

    df = df.merge(u, how="left", on=["movie_title", "d1_date", "date_show"])
    df["comp_size_nat"] = np.log1p(df["comp_size_nat"].fillna(0) / df["movie_title"].map(own_size))
    return df

COMP_COLS = ["comp_size_cin", "comp_fresh_cin", "comp_size_nat", "date_idx", "own_idx", "date_idx_rel"]
OPENING_SHAPE = typical_opening_shape(full[full["is_train"] == 1])
is_tr = full["is_train"] == 1
full = pd.concat([add_competition(full[is_tr], full[is_tr], OPENING_SHAPE),
                  add_competition(full[~is_tr], full[~is_tr], OPENING_SHAPE)], ignore_index=True)
full.groupby("is_train")[COMP_COLS].mean().round(3)

,comp_size_cin,comp_fresh_cin,comp_size_nat,date_idx,own_idx,date_idx_rel
is_train,,,,,,
0,1.301,0.898,1.438,0.005,-0.012,0.020
1,1.256,0.874,1.485,-0.000,0.004,-0.003


### Baris latih tambahan (jendela bergeser)
Hanya ada sekitar dua ratus pembukaan film untuk dipelajari. Menggeser D1 lebih lambat 1 sampai 14 hari memperlakukan hari yang lebih akhir dari film yang sama sebagai D1 dan menghasilkan contoh latih baru dengan tiket nyata, mencakup tahap lain dari masa tayang film dan hari kalender lain.
- age_offset: jumlah hari D1 digeser, 0 untuk baris asli dan baris test, sehingga model dapat membedakan baris hasil geser.
Baris geser hanya dipakai untuk fitting, tidak pernah untuk validasi, dan bobotnya lebih rendah dari baris asli pada model tree karena kurang khas dibanding pembukaan yang sebenarnya.

In [26]:
AUG_SHIFTS = [1, 2, 3, 5, 7, 10, 14]

def build_shifted_rows(shift):
    d1 = train_d1 + pd.Timedelta(days=shift)
    d1 = d1[d1 + pd.Timedelta(days=9) <= train["date_show"].max()]
    window = build_window(train, d1, 10)
    df = to_long(window[window["t3"] > 0].reset_index(drop=True), with_target=True).assign(is_train=1)
    df = drop_gap_rows(join_external(join_sources(df)))
    df, _ = add_genres(add_calendar_flags(df), GENRE_COLS)
    df = add_target(drop_unused(set_categories(df)))
    df = add_weekday_price(add_cinema_features(add_film_features(add_pair_features(add_off_block(df)), window)))
    df = add_film_profile(add_late_start(add_window_offdays(df)))
    df = add_competition(df, full[full["is_train"] == 1], OPENING_SHAPE)
    return df.assign(age_offset=shift)

full["age_offset"] = 0
AUG = pd.concat([build_shifted_rows(s) for s in AUG_SHIFTS], ignore_index=True)
print(f"extra rows: {len(AUG)} ({len(AUG) / (full['is_train'] == 1).sum():.1f}x the real train rows)")
AUG.groupby("age_offset")["target_ratio"].agg(["size", "mean"]).round(3)

extra rows: 254483 (4.7x the real train rows)


,size,mean
age_offset,,
1,50854,0.464
2,50833,0.482
3,48573,0.521
5,33873,0.635
7,28516,0.591
10,26786,0.410
14,15048,0.591


# Validasi Waktu

### Setup
Semua model memprediksi target_ratio, sama seperti notebook utama. Baris train yang menyentuh hari tidak lengkap dihapus, juga dari baris tambahan.

In [27]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

RUN_MODELS = ["kumo", "lgbm", "nn"]
MODEL_SEEDS = [42, 7, 123]
ZERO_SNAP = {"lgbm": 0.2, "nn": 0.2}
USE_EXTRA_ROWS = True
EXTRA_WEIGHT = 0.5

NUM_FEATURES = [
    "t1", "t2", "t3", "show1", "show2", "show3", "horizon", "scale", "price",
    "trend", "t3_share", "show_mean", "tickets_per_show3", "active_days",
    "mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend", "pair_share", "cl_scale", "cl_movies",
    "dow", "d1_dow", "price_ratio",
    "is_holiday", "is_cuti_bersama", "is_school_holiday", "off_block_len",
] + OFF_COLS + [
    "first_active", "eff_age", "last_ratio", "active_mean",
] + COMP_COLS + ["age_offset"] + COMMON_GENRES + PROFILE_COLS
FEATURES = NUM_FEATURES + CAT_COLS

def incomplete_days(raw):
    """days on which train.csv is partial (found by a rule): far fewer shows or cinemas than the days around it (21-day median), or no rows at all"""
    g = raw.groupby("date_show").agg(shows=("total_show", "sum"), cinemas=("cinema_ids", "nunique")).reindex(pd.date_range(raw["date_show"].min(), raw["date_show"].max()))
    rel_shows = g["shows"] / g["shows"].rolling(21, center=True, min_periods=7).median()
    rel_cinemas = g["cinemas"] / g["cinemas"].rolling(21, center=True, min_periods=7).median()
    return pd.DatetimeIndex(g.index[(rel_shows < 0.75) | (rel_cinemas < 0.85) | g["shows"].isna()])

INCOMPLETE_DAYS = incomplete_days(train)

def touches_incomplete(df):
    """True for rows whose target day or any of D1-D3 is an incomplete day: their target and their D1-D3 base are under-reported"""
    return (df["date_show"].isin(INCOMPLETE_DAYS) | np.any([(df["d1_date"] + pd.Timedelta(days=k)).isin(INCOMPLETE_DAYS) for k in range(3)], axis=0)).values

drop_rows = (full["is_train"] == 1) & touches_incomplete(full)
print(f"incomplete train days: {len(INCOMPLETE_DAYS)} ({INCOMPLETE_DAYS.min().date()} .. {INCOMPLETE_DAYS.max().date()}); train rows dropped: {int(drop_rows.sum())} of {int((full['is_train'] == 1).sum())}")
train_df = full[(full["is_train"] == 1) & ~drop_rows].reset_index(drop=True)
test_df = full[full["is_train"] == 0].reset_index(drop=True)
CINEMA_CODE = {c: k for k, c in enumerate(sorted(pd.concat([train_df["cinema_ids"], test_df["cinema_ids"]]).unique()))}
for df_ in (train_df, test_df):
    df_["cinema_code"] = df_["cinema_ids"].map(CINEMA_CODE)
X, y, groups = train_df[FEATURES + ["cinema_code"]], train_df["target_ratio"].values, train_df["movie_title"]
X_test = test_df[FEATURES + ["cinema_code"]]
AUG = AUG[~touches_incomplete(AUG)].reset_index(drop=True)
AUG["cinema_code"] = AUG["cinema_ids"].map(CINEMA_CODE)
X_aug, y_aug = AUG[FEATURES + ["cinema_code"]], AUG["target_ratio"].values

def mase_ratio(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def extra_weights(is_extra):
    return np.where(is_extra, EXTRA_WEIGHT, 1.0)

def seed_average(make_model, X_tr, y_tr, X_pred, prep=lambda d: d, sample_weight=None):
    preds = [make_model(seed).fit(prep(X_tr), y_tr, sample_weight=sample_weight).predict(prep(X_pred)) for seed in MODEL_SEEDS]
    return np.mean(preds, axis=0)

BLEND_SNAP = 0.1
print(f"features: {len(FEATURES)} | train rows: {len(train_df)} | extra rows: {len(AUG)} | test rows: {len(test_df)}")


incomplete train days: 13 (2025-06-01 .. 2025-06-16); train rows dropped: 1766 of 54671
features: 77 | train rows: 52905 | extra rows: 244229 | test rows: 72611


### Lipatan waktu
Film validasi adalah film dengan D1 di satu bulan, dan data latih hanya film yang jendela D1-D10 nya selesai sedikitnya GAP_DAYS sebelum bulan itu mulai, sehingga model selalu memprediksi masa depan. Baris tambahan memakai aturan yang sama, dan film yang jatuh di jeda tidak dipakai di sisi mana pun.

In [28]:
TIME_FOLDS = {
    "juli": ("2025-07-01", "2025-07-31"),
    "agustus": ("2025-08-01", "2025-08-31"),
    "september": ("2025-09-01", "2025-09-30"),
}
GAP_DAYS = 10

def split_time_fold(d1_from, d1_to):
    val_mask = train_df["d1_date"].between(pd.Timestamp(d1_from), pd.Timestamp(d1_to)).values
    cut = pd.Timestamp(d1_from) - pd.Timedelta(days=GAP_DAYS)
    ends_before_cut = lambda df: ((df["d1_date"] + pd.Timedelta(days=9)) < cut).values
    return val_mask, ends_before_cut(train_df), ends_before_cut(AUG), cut

summary = []
for name, (d1_from, d1_to) in TIME_FOLDS.items():
    val_mask, tr_mask, aug_mask, cut = split_time_fold(d1_from, d1_to)
    summary.append({"lipatan": name, "film validasi": train_df.loc[val_mask, "movie_title"].nunique(), "baris validasi": int(val_mask.sum()),
                    "baris latih asli": int(tr_mask.sum()), "baris latih tambahan": int(aug_mask.sum()),
                    "D10 film latih sebelum": str(cut.date()),
                    "mean target validasi": round(float(y[val_mask].mean()), 3), "mean target latih": round(float(y[tr_mask].mean()), 3)})
pd.DataFrame(summary)

,lipatan,film validasi,baris validasi,baris latih asli,baris latih tambahan,D10 film latih sebelum,mean target validasi,mean target latih
0,juli,37,12348,15280,65708,2025-06-21,0.637,0.685
1,agustus,35,11235,25787,111635,2025-07-22,0.557,0.682
2,september,23,8008,37645,171079,2025-08-22,0.536,0.634


# Regressor Models

### LGBM Regressor
Gradient boosted trees dengan objective absolute error yang sesuai dengan metrik. Baris tambahan diberi bobot sampel lebih rendah dari baris asli, dan prediksi beberapa seed dirata-rata agar hasil tidak bergantung pada satu seed yang kebetulan bagus.

In [29]:
LGBM_PARAMS = dict(
    n_estimators=1500,
    learning_rate=0.0149601,
    num_leaves=87,
    min_child_samples=11,
    subsample=0.641476,
    colsample_bytree=0.48309,
    reg_lambda=1.85904,
    reg_alpha=11.5708,
    subsample_freq=1,
)

def lgbm_fit_predict(X_tr, y_tr, is_extra, X_pred):
    X_tr, X_pred = X_tr[FEATURES], X_pred[FEATURES]
    make_model = lambda seed: LGBMRegressor(objective="l1", random_state=seed, verbose=-1, **LGBM_PARAMS)
    return seed_average(make_model, X_tr, y_tr, X_pred, sample_weight=extra_weights(is_extra))


### Bobot Kumo-Tabular
Kumo-Tabular adalah model pretrained dari NVIDIA. Model ini tidak dilatih di sini dan bukan bagian dari model yang diekspor: cell ini mengunduh bobotnya dari Hugging Face, dan tidak melakukan apa pun bila sudah tersimpan di cache.

In [30]:
KUMO_REPO = "nvidia/Kumo-Tabular"
if "kumo" in RUN_MODELS:
    print("Kumo-Tabular weights at", snapshot_download(KUMO_REPO))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Kumo-Tabular weights at C:\Users\leouw\.cache\huggingface\hub\models--nvidia--Kumo-Tabular\snapshots\c5893d418e162b9f841ae179dc767afefff73831


### Kumo-Tabular
Model in-context pretrained: sampel baris train (baris asli dan baris geser) diberikan sebagai konteks dan tidak ada bobot yang diperbarui. Prediksinya adalah median dari quantile yang diprediksi, yang sesuai dengan metrik absolute error. Beberapa konteks acak dan beberapa pass dirata-rata karena satu sampel konteks saja menambah variansi.

In [31]:
KUMO_CONTEXT_ROWS = 9000
KUMO_EXTRA_ROWS = 3000
KUMO_CONTEXT_SEEDS = [42, 7, 123]
KUMO_ESTIMATORS = 8
KUMO_STYPES = {c: "numerical" for c in NUM_FEATURES} | {c: "categorical" for c in CAT_COLS}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
kumo_model = sdm.models.KumoTabular(task="regression", size="large", device=DEVICE).eval() if "kumo" in RUN_MODELS else None

def kumo_fit_predict(X_tr, y_tr, is_extra, X_pred, chunk_rows=2000):
    X_tr, X_pred = X_tr[FEATURES], X_pred[FEATURES]
    real_idx, extra_idx = np.flatnonzero(~is_extra), np.flatnonzero(is_extra)
    preds = []
    bar = tqdm(total=len(KUMO_CONTEXT_SEEDS) * int(np.ceil(len(X_pred) / chunk_rows)), desc="kumo", unit="chunk")
    for seed in KUMO_CONTEXT_SEEDS:
        rng = np.random.default_rng(seed)
        ctx = np.r_[rng.choice(real_idx, min(KUMO_CONTEXT_ROWS, len(real_idx)), replace=False),
                    rng.choice(extra_idx, min(KUMO_EXTRA_ROWS, len(extra_idx)), replace=False)].astype(int)
        x_ctx = TableTensor.from_pandas(X_tr.iloc[ctx], KUMO_STYPES, device=DEVICE)
        y_ctx = TableTensor.from_pandas(pd.DataFrame({"y": y_tr[ctx]}), {"y": "numerical"}, device=DEVICE)
        kumo_model.fit(x_ctx, y_ctx, num_estimators=KUMO_ESTIMATORS, generator=torch.Generator(device=DEVICE).manual_seed(seed))
        quantiles = []
        for start in range(0, len(X_pred), chunk_rows):
            x_q = TableTensor.from_pandas(X_pred.iloc[start:start + chunk_rows], KUMO_STYPES, device=DEVICE)
            quantiles.append(kumo_model.predict(x_q).numerical.float().cpu())
            bar.update(1); bar.set_postfix(seed=seed)
        q = torch.cat(quantiles).numpy()
        preds.append(q[:, q.shape[1] // 2])
        torch.cuda.empty_cache()
    bar.close()
    return np.mean(preds, axis=0)


### Neural network
MLP kecil dengan embedding untuk bioskop dan kolom kategori. Embedding memberi setiap bioskop dan kategori representasi yang dipelajari, sehingga bioskop dengan perilaku mirip saling berdekatan, sesuatu yang tidak bisa ditangkap tree dan Kumo secara langsung. Dilatih dengan weighted L1 loss yang sama dengan metrik (baris tambahan berbobot lebih rendah), lima seed dirata-rata, dan input numerik dipotong pada persentil ekstrem lalu distandarisasi. Kesalahannya berbeda dari LGBM dan Kumo, sehingga melengkapi blend.

In [32]:
NN_SEEDS = [1, 2, 3, 4, 5]
NN_EPOCHS = 15
NN_WIDTH, NN_DEPTH, NN_DROPOUT = 512, 4, 0.287
NN_LR, NN_WEIGHT_DECAY, NN_BATCH, NN_EMB_CAP = 7.9e-3, 6.9e-6, 256, 8
NN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class EmbeddingMLP(torch.nn.Module):
    def __init__(self, n_num, cardinalities, width=NN_WIDTH, depth=NN_DEPTH, dropout=NN_DROPOUT):
        super().__init__()
        self.emb = torch.nn.ModuleList([torch.nn.Embedding(c, min(NN_EMB_CAP, (c + 1) // 2 + 1)) for c in cardinalities])
        dim = n_num + sum(e.embedding_dim for e in self.emb)
        layers = []
        for k in range(depth):
            nxt = width if k == 0 else width // 2
            layers += [torch.nn.Linear(dim, nxt), torch.nn.BatchNorm1d(nxt), torch.nn.SiLU(), torch.nn.Dropout(dropout)]
            dim = nxt
        self.mlp = torch.nn.Sequential(*layers, torch.nn.Linear(dim, 1))

    def forward(self, x_num, x_cat):
        return self.mlp(torch.cat([x_num] + [e(x_cat[:, k]) for k, e in enumerate(self.emb)], dim=1)).squeeze(1)

def nn_prepare(X, st):
    v = X[st["num"]].astype(float)
    z = ((v.fillna(st["med"]).clip(st["lo"], st["hi"], axis=1) - st["mu"]) / st["sd"]).values
    flags = v[st["nan_cols"]].isna().astype(float).values
    codes = np.stack([X[c].cat.codes.values + 1 for c in CAT_COLS] + [X["cinema_code"].values + 1], axis=1)
    return (torch.tensor(np.hstack([z, flags]).astype(np.float32), device=NN_DEVICE), torch.tensor(codes.astype(np.int64), device=NN_DEVICE))

def train_nn(X_tr, y_tr, is_extra):
    num = [c for c in FEATURES if c not in CAT_COLS]
    ref = X_tr[num].astype(float)
    med, lo, hi = ref.median(), ref.quantile(0.01), ref.quantile(0.99)
    clipped = ref.clip(lo, hi, axis=1)
    st = {"num": num, "med": med, "lo": lo, "hi": hi, "mu": clipped.mean(), "sd": clipped.std() + 1e-9,
          "nan_cols": [c for c in num if ref[c].isna().mean() > 0.01],
          "cards": [len(X_tr[c].cat.categories) + 2 for c in CAT_COLS] + [len(CINEMA_CODE) + 2]}
    xn, xc = nn_prepare(X_tr, st)
    target = torch.tensor((y_tr + 0.05).astype(np.float32), device=NN_DEVICE)
    weight = torch.tensor(extra_weights(is_extra).astype(np.float32), device=NN_DEVICE)
    states = []
    for seed in tqdm(NN_SEEDS, desc="nn seeds", leave=False):
        torch.manual_seed(seed)
        net = EmbeddingMLP(xn.shape[1], st["cards"]).to(NN_DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=NN_LR / 1.5, weight_decay=NN_WEIGHT_DECAY)
        steps = NN_EPOCHS * int(np.ceil(len(xn) / NN_BATCH))
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=NN_LR, total_steps=steps)
        for epoch in range(NN_EPOCHS):
            net.train()
            order = torch.randperm(len(xn), device=NN_DEVICE)
            for start in range(0, len(xn), NN_BATCH):
                idx = order[start:start + NN_BATCH]
                if len(idx) < 2:
                    continue
                loss = (weight[idx] * (net(xn[idx], xc[idx]) - target[idx]).abs()).mean()
                opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        states.append({k: v.detach().cpu() for k, v in net.state_dict().items()})
    return {"stats": st, "states": states}

def predict_nn(model, X_pred):
    xn, xc = nn_prepare(X_pred, model["stats"])
    preds = []
    for state in model["states"]:
        net = EmbeddingMLP(xn.shape[1], model["stats"]["cards"]).to(NN_DEVICE)
        net.load_state_dict(state)
        net.eval()
        with torch.no_grad():
            preds.append(net(xn, xc).cpu().numpy())
    return np.mean(preds, axis=0) - 0.05

def nn_fit_predict(X_tr, y_tr, is_extra, X_pred):
    return predict_nn(train_nn(X_tr, y_tr, is_extra), X_pred)


### Latih dan prediksi per lipatan
Setiap model dilatih sekali pada data latih lipatan itu lalu memprediksi film validasinya. Prediksi di bawah ZERO_SNAP dijadikan 0 seperti pada notebook utama.

In [33]:
FIT_PREDICT = {"kumo": kumo_fit_predict, "lgbm": lgbm_fit_predict, "nn": nn_fit_predict}

def run_fold(name):
    val_mask, tr_mask, aug_mask, _ = split_time_fold(*TIME_FOLDS[name])
    X_fit = pd.concat([X[tr_mask], X_aug[aug_mask]], ignore_index=True)
    y_fit = np.r_[y[tr_mask], y_aug[aug_mask]]
    is_extra = np.r_[np.zeros(int(tr_mask.sum()), bool), np.ones(int(aug_mask.sum()), bool)]
    X_val = X[val_mask]
    preds = {}
    for m in RUN_MODELS:
        p = np.clip(FIT_PREDICT[m](X_fit, y_fit, is_extra, X_val), 0, None)
        preds[m] = np.where(p < ZERO_SNAP.get(m, 0.0), 0.0, p)
        print(f"{name} {m}: MASE={mase_ratio(y[val_mask], preds[m]):.4f}")
    return {"y": y[val_mask], "film": groups[val_mask].values, "horizon": train_df.loc[val_mask, "horizon"].values, "preds": preds}

RESULTS = {name: run_fold(name) for name in TIME_FOLDS}

kumo:   0%|          | 0/21 [00:00<?, ?chunk/s]

C:\Users\leouw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdm\tensor\string.py:502: RuntimeWarning: Falling back to a CPU-based string sort because cuDF is not installed. Install cuDF to enable faster CUDA-based string sorting without device synchronization. This warning will be suppressed for the remainder of this process.
  warn_once(
C:\Users\leouw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sdm\relational\join.py:73: RuntimeWarning: Falling back to a CPU-based join because cuDF is not installed. Install cuDF to enable faster CUDA-based joins without device synchronization. This warning will be suppressed for the remainder of this process.
  warn_once(


juli kumo: MASE=0.3520
juli lgbm: MASE=0.3471


nn seeds:   0%|          | 0/5 [00:00<?, ?it/s]

juli nn: MASE=0.3824


...

agustus nn: MASE=0.4591


kumo:   0%|          | 0/15 [00:00<?, ?chunk/s]

september kumo: MASE=0.2400
september lgbm: MASE=0.2537


nn seeds:   0%|          | 0/5 [00:00<?, ?it/s]

september nn: MASE=0.2539


# Perbandingan Blend

### Skor tiap konfigurasi
Pembandingnya Kumo sendiri, Kumo + LGBM, dan Kumo + LGBM + NN pada beberapa bobot, termasuk 0,4 / 0,3 / 0,3 yang dipakai di notebook utama. Rata-rata lipatan adalah rata-rata skor tiga lipatan, dan gabungan adalah MASE seluruh baris validasi. Naif rasio 1 sama dengan menebak rata-rata D1-D3.

In [34]:
def blend(preds, w):
    used = {m: v for m, v in w.items() if v > 0}
    if len(used) == 1:
        return preds[next(iter(used))]
    p = sum(v * preds[m] for m, v in used.items()) / sum(used.values())
    return np.where(p < BLEND_SNAP, 0.0, p)

def pool(names):
    return {"y": np.concatenate([RESULTS[n]["y"] for n in names]),
            "film": np.concatenate([RESULTS[n]["film"] for n in names]),
            "horizon": np.concatenate([RESULTS[n]["horizon"] for n in names]),
            "preds": {m: np.concatenate([RESULTS[n]["preds"][m] for n in names]) for m in RUN_MODELS}}

ALL = pool(list(RESULTS))
BASE_W = {"kumo": 0.4, "lgbm": 0.3, "nn": 0.3}
WEIGHT_CONFIGS = {
    "kumo": {"kumo": 1.0},
    "lgbm": {"lgbm": 1.0},
    "nn": {"nn": 1.0},
    "kumo 0.7 / lgbm 0.3": {"kumo": 0.7, "lgbm": 0.3},
    "kumo 0.5 / lgbm 0.5": {"kumo": 0.5, "lgbm": 0.5},
    "kumo 0.6 / lgbm 0.2 / nn 0.2": {"kumo": 0.6, "lgbm": 0.2, "nn": 0.2},
    "kumo 0.5 / lgbm 0.3 / nn 0.2": {"kumo": 0.5, "lgbm": 0.3, "nn": 0.2},
    "kumo 0.4 / lgbm 0.3 / nn 0.3 (dipakai)": BASE_W,
    "kumo 0.4 / lgbm 0.2 / nn 0.4": {"kumo": 0.4, "lgbm": 0.2, "nn": 0.4},
    "kumo 0.34 / lgbm 0.33 / nn 0.33": {"kumo": 0.34, "lgbm": 0.33, "nn": 0.33},
}

rows = {"naif rasio 1 (D1-D3)": {n: mase_ratio(r["y"], 1.0) for n, r in RESULTS.items()},
        "naif 0": {n: mase_ratio(r["y"], 0.0) for n, r in RESULTS.items()}}
for label, w in WEIGHT_CONFIGS.items():
    rows[label] = {n: mase_ratio(r["y"], blend(r["preds"], w)) for n, r in RESULTS.items()}
comparison = pd.DataFrame(rows).T
comparison["rata-rata lipatan"] = comparison[list(RESULTS)].mean(axis=1)
comparison["gabungan"] = [mase_ratio(ALL["y"], 1.0), mase_ratio(ALL["y"], 0.0)] + [mase_ratio(ALL["y"], blend(ALL["preds"], w)) for w in WEIGHT_CONFIGS.values()]
comparison.round(4)

,juli,agustus,september,rata-rata lipatan,gabungan
naif rasio 1 (D1-D3),0.7255,0.7334,0.6686,0.7092,0.7139
naif 0,0.6369,0.5569,0.5356,0.5764,0.5827
kumo,0.3520,0.3218,0.2400,0.3046,0.3129
lgbm,0.3471,0.3229,0.2537,0.3079,0.3148
nn,0.3824,0.4591,0.2539,0.3651,0.3771
kumo 0.7 / lgbm 0.3,0.3480,0.3191,0.2407,0.3026,0.3105
kumo 0.5 / lgbm 0.5,0.3462,0.3190,0.2444,0.3032,0.3107
kumo 0.6 / lgbm 0.2 / nn 0.2,0.3507,0.3348,0.2372,0.3076,0.3163
kumo 0.5 / lgbm 0.3 / nn 0.2,0.3496,0.3337,0.2381,0.3071,0.3157
kumo 0.4 / lgbm 0.3 / nn 0.3 (dipakai),0.3515,0.3452,0.2380,0.3116,0.3205


### Bobot dipilih di dua lipatan lalu diuji di lipatan ketiga
Untuk tiap lipatan, bobot terbaik dicari di dua lipatan lainnya (grid 0,05 untuk Kumo + LGBM dan grid 0,1 untuk tiga model), lalu dipakai pada lipatan itu. Ini ukuran yang jujur karena bobot tidak dipilih pada data yang menilainya.

In [35]:
def best_weights(res, models, step):
    grid = np.round(np.arange(0, 1 + 1e-9, step), 4)
    cands = [dict(zip(models, c)) for c in np.array(np.meshgrid(*[grid] * len(models))).T.reshape(-1, len(models)) if abs(c.sum() - 1) < 1e-9]
    return sorted(((mase_ratio(res["y"], blend(res["preds"], w)), w) for w in cands), key=lambda t: t[0])

rows = []
for held in RESULTS:
    others = [n for n in RESULTS if n != held]
    train_pool = pool(others)
    for label, models, step in (("kumo + lgbm", ["kumo", "lgbm"], 0.05), ("kumo + lgbm + nn", ["kumo", "lgbm", "nn"], 0.1)):
        w = best_weights(train_pool, models, step)[0][1]
        rows.append({"lipatan uji": held, "konfigurasi": label, "bobot dari": "+".join(others), "bobot": {k: v for k, v in w.items() if v > 0},
                     "MASE uji": round(mase_ratio(RESULTS[held]["y"], blend(RESULTS[held]["preds"], w)), 4),
                     "MASE kumo": round(mase_ratio(RESULTS[held]["y"], RESULTS[held]["preds"]["kumo"]), 4),
                     "MASE 0.4/0.3/0.3": round(mase_ratio(RESULTS[held]["y"], blend(RESULTS[held]["preds"], BASE_W)), 4)})
pd.DataFrame(rows)

,lipatan uji,konfigurasi,bobot dari,bobot,MASE uji,MASE kumo,MASE 0.4/0.3/0.3
0,juli,kumo + lgbm,agustus+september,"{'kumo': 0.75, 'lgbm': 0.25}",0.3488,0.3520,0.3515
1,juli,kumo + lgbm + nn,agustus+september,"{'kumo': 0.7, 'lgbm': 0.3}",0.3480,0.3520,0.3515
2,agustus,kumo + lgbm,juli+september,"{'kumo': 0.65, 'lgbm': 0.35}",0.3189,0.3218,0.3452
3,agustus,kumo + lgbm + nn,juli+september,"{'kumo': 0.6, 'lgbm': 0.3, 'nn': 0.1}",0.3245,0.3218,0.3452
4,september,kumo + lgbm,juli+agustus,"{'kumo': 0.4, 'lgbm': 0.6}",0.2460,0.2400,0.2380
5,september,kumo + lgbm + nn,juli+agustus,"{'kumo': 0.4, 'lgbm': 0.6}",0.2460,0.2400,0.2380


### Grid bobot blend tiga model
Sepuluh kombinasi bobot terbaik pada seluruh baris validasi gabungan. Bobot dipilih pada data yang sama dengan yang menilainya sehingga skornya optimis, dan dipakai hanya untuk melihat apakah selisih antar bobot cukup besar dibanding noise.

In [36]:
top = best_weights(ALL, ["kumo", "lgbm", "nn"], 0.1)[:10]
pd.DataFrame([{**w, "MASE gabungan": round(s, 4)} for s, w in top])

,kumo,lgbm,nn,MASE gabungan
0,0.6,0.4,0.0,0.3104
1,0.7,0.3,0.0,0.3105
2,0.5,0.5,0.0,0.3107
3,0.4,0.6,0.0,0.3111
4,0.8,0.2,0.0,0.3112
5,0.3,0.7,0.0,0.3117
6,0.5,0.4,0.1,0.3119
7,0.6,0.3,0.1,0.3121
8,0.4,0.5,0.1,0.3122
9,0.9,0.1,0.0,0.3122


### Bootstrap per film
Film validasi diambil ulang dengan pengembalian sebanyak 2000 kali, lalu selisih MASE terhadap Kumo sendiri dihitung pada sampel yang sama. Selisih negatif berarti blend lebih baik dari Kumo, dan interval yang tidak melewati 0 berarti selisihnya konsisten antar film.

In [37]:
N_BOOT = 2000

def bootstrap_vs_kumo(res, w):
    films = pd.factorize(res["film"])[0]
    n_films = films.max() + 1
    counts = np.bincount(films, minlength=n_films)
    film_error = lambda p: np.bincount(films, weights=np.abs(res["y"] - p), minlength=n_films)
    diff = film_error(blend(res["preds"], w)) - film_error(res["preds"]["kumo"])
    draws = np.random.default_rng(SEED).integers(0, n_films, size=(N_BOOT, n_films))
    boot = diff[draws].sum(axis=1) / counts[draws].sum(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"film": n_films, "selisih MASE vs kumo": round(diff.sum() / counts.sum(), 4), "CI 95%": f"[{lo:.4f}, {hi:.4f}]", "P(lebih baik dari kumo)": round(float((boot < 0).mean()), 3)}

CANDIDATES = {"kumo 0.7 / lgbm 0.3": {"kumo": 0.7, "lgbm": 0.3}, "kumo 0.4 / lgbm 0.3 / nn 0.3": BASE_W}
rows = []
for name, res in {**RESULTS, "gabungan": ALL}.items():
    for label, w in CANDIDATES.items():
        rows.append({"lipatan": name, "konfigurasi": label, **bootstrap_vs_kumo(res, w)})
pd.DataFrame(rows)

,lipatan,konfigurasi,film,selisih MASE vs kumo,CI 95%,P(lebih baik dari kumo)
0,juli,kumo 0.7 / lgbm 0.3,37,-0.0040,"[-0.0092, 0.0000]",0.973
1,juli,kumo 0.4 / lgbm 0.3 / nn 0.3,37,-0.0005,"[-0.0130, 0.0101]",0.529
2,agustus,kumo 0.7 / lgbm 0.3,35,-0.0027,"[-0.0061, 0.0005]",0.952
3,agustus,kumo 0.4 / lgbm 0.3 / nn 0.3,35,0.0234,"[0.0024, 0.0464]",0.013
4,september,kumo 0.7 / lgbm 0.3,23,0.0007,"[-0.0039, 0.0043]",0.319
5,september,kumo 0.4 / lgbm 0.3 / nn 0.3,23,-0.0019,"[-0.0120, 0.0064]",0.607
6,gabungan,kumo 0.7 / lgbm 0.3,95,-0.0024,"[-0.0048, -0.0000]",0.977
7,gabungan,kumo 0.4 / lgbm 0.3 / nn 0.3,95,0.0076,"[-0.0021, 0.0184]",0.066


### Korelasi error antar model
Error bertanda (prediksi dikurangi aktual) pada baris validasi. Korelasi yang rendah berarti kesalahan ketiga model berbeda sehingga saling menutupi saat dirata-ratakan.

In [38]:
for name, res in {**RESULTS, "gabungan": ALL}.items():
    err = pd.DataFrame({m: p - res["y"] for m, p in res["preds"].items()})
    print(name, "error bertanda")
    print(err.corr().round(3))
    print(name, "besar error")
    print(err.abs().corr().round(3))

juli error bertanda
       kumo   lgbm     nn
kumo  1.000  0.990  0.962
lgbm  0.990  1.000  0.959
nn    0.962  0.959  1.000
juli besar error
       kumo   lgbm     nn
kumo  1.000  0.990  0.965
lgbm  0.990  1.000  0.963
nn    0.965  0.963  1.000
agustus error bertanda
       kumo   lgbm     nn
kumo  1.000  0.967  0.809
lgbm  0.967  1.000  0.768
nn    0.809  0.768  1.000
agustus besar error
       kumo   lgbm     nn
kumo  1.000  0.964  0.807
lgbm  0.964  1.000  0.792
nn    0.807  0.792  1.000
september error bertanda
       kumo   lgbm     nn
kumo  1.000  0.954  0.899
lgbm  0.954  1.000  0.873
nn    0.899  0.873  1.000
september besar error
       kumo   lgbm     nn
kumo  1.000  0.946  0.903
lgbm  0.946  1.000  0.876
nn    0.903  0.876  1.000
gabungan error bertanda
       kumo   lgbm     nn
kumo  1.000  0.980  0.902
lgbm  0.980  1.000  0.885
nn    0.902  0.885  1.000
gabungan besar error
       kumo   lgbm     nn
kumo  1.000  0.980  0.911
lgbm  0.980  1.000  0.903
nn    0.911  0.903  1.

### Skor per horizon
MASE tiap model dan blend 0,4 / 0,3 / 0,3 pada tiap horizon D4-D10, dari seluruh baris validasi gabungan.

In [39]:
table = {m: pd.Series(np.abs(ALL["y"] - p)).groupby(ALL["horizon"]).mean() for m, p in ALL["preds"].items()}
table["blend 0.4/0.3/0.3"] = pd.Series(np.abs(ALL["y"] - blend(ALL["preds"], BASE_W))).groupby(ALL["horizon"]).mean()
pd.DataFrame(table).round(4)

,kumo,lgbm,nn,blend 0.4/0.3/0.3
4,0.3711,0.3669,0.4017,0.3616
5,0.3499,0.3567,0.4088,0.3585
6,0.3015,0.3017,0.3752,0.3136
7,0.3074,0.3089,0.3771,0.3178
8,0.2769,0.2838,0.3454,0.2900
9,0.2765,0.2789,0.3472,0.2872
10,0.3067,0.3069,0.3842,0.3146
